# Chin Masking with MediaPipe
This notebook detects face landmarks and applies a smooth chin mask overlay using MediaPipe.

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from scipy.interpolate import splprep, splev
import matplotlib.pyplot as plt

In [ ]:
# Initialize MediaPipe Face Landmarker
print("Loading MediaPipe model...")
fm_model_path = 'face_landmarker.task'
fm_base_options = python.BaseOptions(model_asset_path=fm_model_path)
fm_options = vision.FaceLandmarkerOptions(base_options=fm_base_options,
                                       output_face_blendshapes=False,
                                       output_facial_transformation_matrixes=False,
                                       num_faces=1)
landmarker = vision.FaceLandmarker.create_from_options(fm_options)
print("MediaPipe ready!")

In [ ]:
import tkinter as tk
from tkinter import filedialog

print("Opening file dialog... Please check your taskbar for the popup window.")

root = tk.Tk()
root.attributes('-topmost', True) # Bring to front
root.withdraw() # Hide the main tk window

file_path = filedialog.askopenfilename(
    title="Select an Image",
    filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp")]
)

root.destroy()

if file_path:
    print(f"Selected file: {file_path}")
else:
    print("No file selected.")
    file_path = None

In [ ]:
import os
import urllib.request

if file_path:
    image = cv2.imread(file_path)
else:
    print("No image provided. Using sample image.")
    IMAGE_PATH = 'sample_face.jpg'
    if not os.path.exists(IMAGE_PATH):
        print("Downloading sample image...")
        req = urllib.request.Request('https://upload.wikimedia.org/wikipedia/commons/thumb/a/a0/Pierre-Person.jpg/800px-Pierre-Person.jpg', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response, open(IMAGE_PATH, 'wb') as out_file:
            out_file.write(response.read())
    image = cv2.imread(IMAGE_PATH)

if image is not None:
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w, _ = image_rgb.shape
    print(f"Image loaded: {w}x{h}")
else:
    print("Failed to load image.")

In [ ]:
# Detect face landmarks and extract lips

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=image_rgb
)

detection_result = landmarker.detect(mp_image)

if not detection_result.face_landmarks:
    raise Exception("No face detected.")

landmarks = detection_result.face_landmarks[0]

# ----------------------------
# Image size
# ----------------------------
h, w = image_rgb.shape[:2]

# ----------------------------
# MediaPipe Lip Landmarks
# ----------------------------

OUTER_LIPS = [
    61,146,91,181,84,17,314,405,321,375,
    291,308,324,318,402,317,14,87,178,88,
    95,78
]

INNER_LIPS = [
    78,191,80,81,82,13,312,311,310,415,
    308,324,318,402,317,14,87,178,88,95
]

# ----------------------------
# Convert to pixel coordinates
# ----------------------------

outer_pts = np.array([
    [int(landmarks[i].x * w), int(landmarks[i].y * h)]
    for i in OUTER_LIPS
], dtype=np.int32)

inner_pts = np.array([
    [int(landmarks[i].x * w), int(landmarks[i].y * h)]
    for i in INNER_LIPS
], dtype=np.int32)

# ----------------------------
# Create lip mask
# ----------------------------

mask = np.zeros((h, w), dtype=np.uint8)

# Outer lips
cv2.fillPoly(mask, [outer_pts], 255)

# Remove mouth opening
cv2.fillPoly(mask, [inner_pts], 0)

# ----------------------------
# Extract lips
# ----------------------------

lips = cv2.bitwise_and(image_rgb, image_rgb, mask=mask)

# ----------------------------
# Crop tightly
# ----------------------------

x, y, ww, hh = cv2.boundingRect(outer_pts)

padding = 8

x = max(0, x - padding)
y = max(0, y - padding)

ww = min(w - x, ww + padding * 2)
hh = min(h - y, hh + padding * 2)

cropped_lips = lips[y:y+hh, x:x+ww]
cropped_mask = mask[y:y+hh, x:x+ww]

# Optional: white background instead of black
white = np.full_like(cropped_lips, 255)
white[cropped_mask > 0] = cropped_lips[cropped_mask > 0]

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(image_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(cropped_mask, cmap="gray")
plt.title("Lip Mask")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(white)
plt.title("Extracted Lips")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Display result
plt.figure(figsize=(8, 8))
plt.imshow(final_image)
plt.axis("off")
plt.title("MediaPipe Chin Overlay")
plt.tight_layout()
plt.show()